# Tratamento do dataset para clustering no Weka

Este notebook faz o tratamento basico do dataset e exporta uma versao limpa para o Weka. No final, ele tambem gera uma versao numerica opcional para algoritmos que precisem de atributos numericos.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

In [2]:
dataset_path = Path('datasets/dataset-tsc.csv')
dataset = pd.read_csv(dataset_path)
dataset.head()

,Nome de usuário,Seu nome,Idade,Altura,Sexo,Cor da pele,Cabelo1,Cabelo2,Usa óculos/lente,Gênero de Filme ou Série favorito,...,Já tem uma graduação,Está em um relacionamento afetivo? (Não é Estado Civil),"Estagiando/trabalhando (qualquer atividade remunerada conta, Ex.: Bolsista PET)",Estado Civil,Cidade de Origem (Sem estado),Mora com os pais,Tem irmãos/irmãs,Tem animal de estimação,Tem CNH (Carteira Nacional de Habilitação),Dirige ou pilota veículos automotores
0,vitorcorreia@aluno.uespi.br,VITOR DOS SANTOS CORREIA,18 - 20,"1,71 - 1,80",Masculino,Branca,Curto,Ondulado,Sim,Ficção Científica,...,Não,Não,Sim,Solteiro(a),São Paulo,Sim,Sim,Sim,Sim,Sim
1,ericpatricio@aluno.uespi.br,ERIC SILVA PATRICIO,18 - 20,"1,71 - 1,80",Masculino,Parda,Curto,Cacheado/Encaracolado,Sim,Romance,...,Não,Sim,Sim,Solteiro(a),Água Doce,Sim,Sim,Sim,Não,Não
2,willamyjosueserejo@gmail.com,WILLAMY JOSUE SANTOS SEREJO,> 24,"1,61, - 1,70",Masculino,Negra,Curto,Cacheado/Encaracolado,Não,Terror,...,Sim,Sim,Sim,Solteiro(a),Parnaíba,Não,Sim,Não,Não,Sim
3,ivanildoaraujo@aluno.uespi.br,IVANILDO DOS SANTOS ARAUJO,> 24,"1,61, - 1,70",Masculino,Parda,Curto,Liso,Não,Terror,...,Não,Sim,Não,Solteiro(a),Parnaiba,Sim,Sim,Sim,Não,Sim
4,samuelnascimento@aluno.uespi.br,SAMUEL DA PENHA NASCIMENTO,18 - 20,">1,80",Masculino,Parda,Curto,Ondulado,Não,Animação,...,Não,Não,Sim,Solteiro(a),Parnaíba,Sim,Sim,Sim,Sim,Sim


In [3]:
def strip_accents(text):
    text = unicodedata.normalize('NFKD', str(text))
    return ''.join(char for char in text if not unicodedata.combining(char))

def normalize_spaces(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

def normalize_token(text):
    text = normalize_spaces(text)
    text = strip_accents(text).lower()
    text = text.replace('/', ' ')
    text = text.replace('-', ' ')
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def slugify(text):
    text = normalize_token(text).replace(' ', '_')
    return text or 'valor'

def to_arff(df, relation_name, output_path):
    lines = [f'@RELATION {relation_name}', '']
    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            lines.append(f'@ATTRIBUTE {column} NUMERIC')
        else:
            values = sorted({slugify(value) for value in df[column].dropna().astype(str)})
            lines.append(f"@ATTRIBUTE {column} {{{','.join(values)}}}")
    lines.append('')
    lines.append('@DATA')
    for row in df.itertuples(index=False, name=None):
        formatted = []
        for value in row:
            if isinstance(value, str):
                formatted.append(slugify(value))
            else:
                formatted.append(str(value))
        lines.append(','.join(formatted))
    output_path.write_text('\n'.join(lines), encoding='utf-8')

In [4]:
clean = dataset.copy()

for column in clean.columns:
    clean[column] = clean[column].astype(str).map(normalize_spaces)

drop_columns = [
    'Nome de usuário',
    'Seu nome',
    'Cabelo1',
]
clean = clean.drop(columns=drop_columns)

language_map = {
    'python': 'Python',
    'java': 'Java',
    'typescript': 'TypeScript',
    'c': 'C',
    'go': 'Go',
    'lua': 'Lua',
}

music_map = {
    'ecletico': 'Ecletico',
    'eurobeat': 'Eurobeat',
    'forro': 'Forro',
    'gospel': 'Gospel',
    'hip hop': 'Hip Hop',
    'mpb': 'MPB',
    'pop': 'Pop',
    'rap': 'Rap',
    'rock': 'Rock',
    'sertanejo': 'Sertanejo',
    'trap rap': 'Trap Rap',
}

ide_map = {
    'intellij': 'IntelliJ',
    'nvim': 'Nvim',
    'vs code': 'VS Code',
    'vscode': 'VS Code',
    'visual studio': 'Visual Studio',
    'visual studio code': 'VS Code',
    'bloco de notas': 'Bloco de Notas',
}

browser_map = {
    'brave': 'Brave',
    'chrome': 'Chrome',
    'google chrome': 'Chrome',
    'edge': 'Edge',
    'microsoft edge': 'Edge',
    'firefox': 'Firefox',
    'opera gx': 'Opera GX',
}

clean['Gênero/Estilo musical favorito'] = clean['Gênero/Estilo musical favorito'].map(
    lambda value: music_map.get(normalize_token(value), value)
)
clean['Linguagem de Programação favorita'] = clean['Linguagem de Programação favorita'].map(
    lambda value: language_map.get(normalize_token(value), value)
)
clean['IDE(Ambiente de Desenvolvimento Integrado) favorito'] = clean[
    'IDE(Ambiente de Desenvolvimento Integrado) favorito'
].map(lambda value: ide_map.get(normalize_token(value), value))
clean['Navegador favorito'] = clean['Navegador favorito'].map(
    lambda value: browser_map.get(normalize_token(value), value)
)
clean['Sistema Operacional Mobile utilizado'] = clean['Sistema Operacional Mobile utilizado'].replace({'IOS': 'iOS'})
clean['Cidade de Origem (Sem estado)'] = clean['Cidade de Origem (Sem estado)'].map(normalize_spaces)

clean.columns = [slugify(column) for column in clean.columns]
clean.head()

,idade,altura,sexo,cor_da_pele,cabelo2,usa_oculos_lente,genero_de_filme_ou_serie_favorito,genero_estilo_musical_favorito,hobby_passatempo_favorito,linguagem_de_programacao_favorita,...,ja_tem_uma_graduacao,esta_em_um_relacionamento_afetivo_nao_e_estado_civil,estagiando_trabalhando_qualquer_atividade_remunerada_conta_ex_bolsista_pet,estado_civil,cidade_de_origem_sem_estado,mora_com_os_pais,tem_irmaos_irmas,tem_animal_de_estimacao,tem_cnh_carteira_nacional_de_habilitacao,dirige_ou_pilota_veiculos_automotores
0,18 - 20,"1,71 - 1,80",Masculino,Branca,Ondulado,Sim,Ficção Científica,Trap Rap,Assistir séries,Java,...,Não,Não,Sim,Solteiro(a),São Paulo,Sim,Sim,Sim,Sim,Sim
1,18 - 20,"1,71 - 1,80",Masculino,Parda,Cacheado/Encaracolado,Sim,Romance,MPB,Jogar,Python,...,Não,Sim,Sim,Solteiro(a),Água Doce,Sim,Sim,Sim,Não,Não
2,> 24,"1,61, - 1,70",Masculino,Negra,Cacheado/Encaracolado,Não,Terror,Pop,Cantar,Python,...,Sim,Sim,Sim,Solteiro(a),Parnaíba,Não,Sim,Não,Não,Sim
3,> 24,"1,61, - 1,70",Masculino,Parda,Liso,Não,Terror,Rock,Games,TypeScript,...,Não,Sim,Não,Solteiro(a),Parnaiba,Sim,Sim,Sim,Não,Sim
4,18 - 20,">1,80",Masculino,Parda,Ondulado,Não,Animação,Pop,Instrumentos,Java,...,Não,Não,Sim,Solteiro(a),Parnaíba,Sim,Sim,Sim,Sim,Sim


In [5]:
output_csv = Path('datasets/dataset-tsc-clean.csv')
output_arff = Path('datasets/dataset-tsc-clean.arff')

clean.to_csv(output_csv, index=False)
to_arff(clean, 'dataset_tsc_clean', output_arff)

print(f'Arquivo CSV limpo: {output_csv}')
print(f'Arquivo ARFF limpo: {output_arff}')
print(f'Formato final: {clean.shape[0]} linhas x {clean.shape[1]} atributos')

Arquivo CSV limpo: datasets/dataset-tsc-clean.csv
Arquivo ARFF limpo: datasets/dataset-tsc-clean.arff
Formato final: 30 linhas x 27 atributos


In [6]:
numeric = clean.copy()

idade_map = {'18 - 20': 1, '21 - 22': 2, '23 -24': 3, '> 24': 4}
altura_map = {'1,50 - 1,60': 1, '1,61, - 1,70': 2, '1,71 - 1,80': 3, '>1,80': 4}

binary_columns = [
    'usa_oculos_lente',
    'vai_se_formar_esse_ano_por_voce_sem_contar_decisao_de_professores',
    'terminou_o_terceiro_ano_em_escola_publica',
    'ja_tem_uma_graduacao',
    'esta_em_um_relacionamento_afetivo_nao_e_estado_civil',
    'estagiando_trabalhando_qualquer_atividade_remunerada_conta_ex_bolsista_pet',
    'mora_com_os_pais',
    'tem_irmaos_irmas',
    'tem_animal_de_estimacao',
    'tem_cnh_carteira_nacional_de_habilitacao',
    'dirige_ou_pilota_veiculos_automotores',
]

numeric['idade'] = numeric['idade'].map(idade_map)
numeric['altura'] = numeric['altura'].map(altura_map)
for column in binary_columns:
    numeric[column] = numeric[column].map({'Não': 0, 'Sim': 1})
numeric['sexo'] = numeric['sexo'].map({'Feminino': 0, 'Masculino': 1})
numeric['estado_civil'] = numeric['estado_civil'].map({'Casado(a)': 1, 'Solteiro(a)': 0})

categorical_columns = numeric.select_dtypes(include=['object', 'string']).columns
encoded = pd.get_dummies(numeric, columns=categorical_columns, dtype=int)
encoded = encoded.loc[:, encoded.nunique() > 1]
encoded.columns = [slugify(column) for column in encoded.columns]

numeric_csv = Path('datasets/dataset-tsc-clustering.csv')
numeric_arff = Path('datasets/dataset-tsc-clustering.arff')

encoded.to_csv(numeric_csv, index=False)
to_arff(encoded, 'dataset_tsc_clustering', numeric_arff)

print(f'Versao numerica opcional: {numeric_csv}')
print(f'ARFF numerico opcional: {numeric_arff}')
print(f'Formato numerico: {encoded.shape[0]} linhas x {encoded.shape[1]} atributos')

Versao numerica opcional: datasets/dataset-tsc-clustering.csv
ARFF numerico opcional: datasets/dataset-tsc-clustering.arff
Formato numerico: 30 linhas x 98 atributos
